In [1]:
import os
import dotenv
dotenv.load_dotenv()
import pandas as pd
from pyogrio.errors import DataSourceError
import geopandas as gpd
from google.cloud import storage

In [2]:
OUTPUT_PATH = os.getenv("CSO_DATA_OUTPUT")
if OUTPUT_PATH is None:
    raise ValueError("Output Path not set")

In [ ]:
try:
    output_file = gpd.read_file(OUTPUT_PATH)
except DataSourceError:
    output_file = gpd.GeoDataFrame({
        "geometry":[],
        "Monitor_ID":[],
        "Event_Start":[],
        "EventEnd":[],
        })
    print("Output file doesn't exist, making a new GeoPandas DataFrame")

## NWG Data

In [4]:
INPUT_PATH = os.getenv("CSO_DATA_LOC")
if INPUT_PATH is None:
    raise ValueError("Input Path not set")

In [20]:
bucketName = INPUT_PATH.split("/")[2]
path_in_bucket = '/'.join(INPUT_PATH.split("/")[3:])

In [17]:
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucketName)

In [39]:
geojson_generator = filter(lambda x: x.name.endswith(".geojson"),bucket.list_blobs(max_results=100,prefix=path_in_bucket))

### Check if geometry remains same for all IDs

In [37]:
filename = next(geojson_generator)

In [44]:
concatSeries = gpd.GeoSeries()

In [ ]:
for row in gpd.read_file("gs://"+bucketName+'/'+filename.name).set_index("id")["geometry"]:
    print(row)
